# Error and Accuracy Metrics in Machine Learning


This notebook covers the formulas, definitions, and code implementation of every major regression and classification metric used to evaluate machine learning models. Each section includes the formula, an explanation, manual computation from scratch, and the equivalent sklearn function.


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, log_loss, classification_report
)
import warnings
warnings.filterwarnings('ignore')
print("Imports ready")

# Regression Error Metrics

Regression metrics measure the distance between predicted continuous values and actual continuous values.


In [ ]:
# Sample data for regression metrics
np.random.seed(42)
y_true_reg = np.array([100, 150, 200, 250, 300, 180, 220, 90, 310, 175])
y_pred_reg = np.array([110, 140, 195, 260, 290, 200, 210, 95, 330, 165])

print("Actual values:   ", y_true_reg)
print("Predicted values:", y_pred_reg)
print("Errors:          ", y_true_reg - y_pred_reg)

### 1.1 Mean Absolute Error (MAE)

**Definition:** The average of the absolute differences between predicted and actual values.

**Formula:** MAE = (1/n) x sum(|y - y_hat|)

In [ ]:
# Manual computation
errors = y_true_reg - y_pred_reg
mae_manual = np.mean(np.abs(errors))
print(f"MAE (manual):   {mae_manual:.4f}")

# sklearn
mae_sklearn = mean_absolute_error(y_true_reg, y_pred_reg)
print(f"MAE (sklearn):  {mae_sklearn:.4f}")

print()
print("Interpretation: on average, predictions are off by this many units,")
print("in the same unit as the original target variable.")

### 1.2 Mean Squared Error (MSE)

**Definition:** The average of the squared differences between predicted and actual values. Squaring penalizes larger errors more heavily.

**Formula:** MSE = (1/n) x sum((y - y_hat)^2)

In [ ]:
# Manual computation
mse_manual = np.mean(errors ** 2)
print(f"MSE (manual):   {mse_manual:.4f}")

# sklearn
mse_sklearn = mean_squared_error(y_true_reg, y_pred_reg)
print(f"MSE (sklearn):  {mse_sklearn:.4f}")

print()
print("Note: MSE is in squared units, which makes direct interpretation harder.")
print("It is mainly used internally by models during training (as the loss function).")

### 1.3 Root Mean Squared Error (RMSE)

**Definition:** The square root of MSE. Brings the error back into the original unit of the target.

**Formula:** RMSE = sqrt(MSE)

In [ ]:
# Manual computation
rmse_manual = np.sqrt(mse_manual)
print(f"RMSE (manual):  {rmse_manual:.4f}")

# sklearn does not have a direct rmse function -- take sqrt of MSE
rmse_sklearn = np.sqrt(mean_squared_error(y_true_reg, y_pred_reg))
print(f"RMSE (sklearn): {rmse_sklearn:.4f}")

print()
print(f"Compare: MAE = {mae_manual:.4f}  vs  RMSE = {rmse_manual:.4f}")
print("RMSE is always >= MAE. The gap grows larger when errors are inconsistent")
print("(some very large, some very small), since RMSE punishes large errors more.")

### 1.4 R-Squared (Coefficient of Determination)

**Definition:** The proportion of variance in the target explained by the model, relative to simply predicting the mean every time.

**Formula:** R-squared = 1 - (SS_residual / SS_total)

In [ ]:
# Manual computation
ss_residual = np.sum((y_true_reg - y_pred_reg) ** 2)
ss_total = np.sum((y_true_reg - np.mean(y_true_reg)) ** 2)
r2_manual = 1 - (ss_residual / ss_total)
print(f"R-squared (manual):  {r2_manual:.4f}")

# sklearn
r2_sklearn = r2_score(y_true_reg, y_pred_reg)
print(f"R-squared (sklearn): {r2_sklearn:.4f}")

print()
print(f"Interpretation: the model explains {r2_manual*100:.1f}% of the variance in the target.")
print("R-squared of 1.0 = perfect fit. R-squared of 0.0 = no better than predicting the mean.")

In [ ]:
# Demonstrating the baseline: predicting the mean every time gives R-squared = 0
baseline_pred = np.full_like(y_true_reg, np.mean(y_true_reg), dtype=float)
r2_baseline = r2_score(y_true_reg, baseline_pred)
print(f"R-squared when always predicting the mean: {r2_baseline:.4f}")
print("This confirms the baseline behavior of R-squared.")

### 1.5 Mean Absolute Percentage Error (MAPE)

**Definition:** The average absolute percentage difference between predicted and actual values. Scale-independent.

**Formula:** MAPE = (100/n) x sum(|(y - y_hat) / y|)

In [ ]:
# Manual computation
mape_manual = np.mean(np.abs((y_true_reg - y_pred_reg) / y_true_reg)) * 100
print(f"MAPE (manual): {mape_manual:.2f}%")

print()
print("Caution: MAPE is undefined when any actual value (y) equals zero,")
print("since it involves dividing by y. Avoid MAPE on data where the target can be zero.")

### 1.6 Visualizing Regression Errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
axes[0].scatter(y_true_reg, y_pred_reg, color='steelblue', s=60)
lims = [min(y_true_reg.min(), y_pred_reg.min()), max(y_true_reg.max(), y_pred_reg.max())]
axes[0].plot(lims, lims, 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = y_true_reg - y_pred_reg
axes[1].bar(range(len(residuals)), residuals, color='coral')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Observation Index')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residuals per Observation')

plt.tight_layout()
plt.show()

### 1.7 Regression Metrics Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R-squared', 'MAPE'],
    'Value': [mae_manual, mse_manual, rmse_manual, r2_manual, mape_manual],
    'Unit': ['Same as target', 'Squared unit', 'Same as target', 'Unitless (0-1)', 'Percentage'],
    'Sensitive to Outliers': ['Low', 'High', 'High', 'Moderate', 'Moderate']
})
print(summary.to_string(index=False))

## Classification Metrics

Classification metrics evaluate how well a model assigns observations to categories. They are built on the four outcomes of a confusion matrix.


In [ ]:
# Sample binary classification data
# 1 = positive class (e.g., disease present), 0 = negative class
y_true_clf = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1])
y_pred_clf = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1])

# Predicted probabilities (needed for ROC-AUC and log loss)
y_prob_clf = np.array([0.91, 0.12, 0.45, 0.83, 0.20, 0.77, 0.61, 0.08,
                        0.95, 0.30, 0.40, 0.88, 0.15, 0.55, 0.70])

print("Actual:       ", y_true_clf)
print("Predicted:    ", y_pred_clf)
print("Probabilities:", y_prob_clf)

### 2.1 The Confusion Matrix

**Definition:** A table comparing predicted labels against actual labels, broken into True Positive, True Negative, False Positive, and False Negative.

In [ ]:
cm = confusion_matrix(y_true_clf, y_pred_clf)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print()
print(f"True Positives (TP):  {tp}  -- correctly predicted positive")
print(f"True Negatives (TN):  {tn}  -- correctly predicted negative")
print(f"False Positives (FP): {fp}  -- incorrectly predicted positive (Type I error)")
print(f"False Negatives (FN): {fn}  -- incorrectly predicted negative (Type II error)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true_clf, y_pred_clf,
    display_labels=['Negative', 'Positive'],
    cmap='Blues', ax=ax
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

### 2.2 Accuracy

**Definition:** The proportion of total predictions that were correct.

**Formula:** Accuracy = (TP + TN) / (TP + TN + FP + FN)

In [ ]:
# Manual computation
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
print(f"Accuracy (manual):  {accuracy_manual:.4f}")

# sklearn
accuracy_sklearn = accuracy_score(y_true_clf, y_pred_clf)
print(f"Accuracy (sklearn): {accuracy_sklearn:.4f}")

print()
print("Caution: accuracy is misleading on imbalanced datasets.")
print("Example below demonstrates this.")

In [ ]:
# Demonstrating the accuracy trap on imbalanced data
imbalanced_true = np.array([0]*95 + [1]*5)       # 95% negative, 5% positive
imbalanced_pred = np.array([0]*100)               # model always predicts negative

acc_imbalanced = accuracy_score(imbalanced_true, imbalanced_pred)
print(f"A model that always predicts the majority class achieves accuracy: {acc_imbalanced:.2f}")
print("This looks excellent but the model never correctly identifies the minority class.")
print("This is why accuracy alone is insufficient for imbalanced problems.")

### 2.3 Precision

**Definition:** Of all instances predicted positive, how many were actually positive.

**Formula:** Precision = TP / (TP + FP)

In [ ]:
precision_manual = tp / (tp + fp)
print(f"Precision (manual):  {precision_manual:.4f}")

precision_sklearn = precision_score(y_true_clf, y_pred_clf)
print(f"Precision (sklearn): {precision_sklearn:.4f}")

print()
print("Use precision when false positives are costly.")
print("Example: a spam filter marking a legitimate email as spam.")

### 2.4 Recall (Sensitivity)

**Definition:** Of all instances that were actually positive, how many did the model correctly identify.

**Formula:** Recall = TP / (TP + FN)

In [ ]:
recall_manual = tp / (tp + fn)
print(f"Recall (manual):  {recall_manual:.4f}")

recall_sklearn = recall_score(y_true_clf, y_pred_clf)
print(f"Recall (sklearn): {recall_sklearn:.4f}")

print()
print("Use recall when false negatives are costly.")
print("Example: failing to detect a patient who actually has a disease.")

### 2.5 F1 Score

**Definition:** The harmonic mean of precision and recall, providing a single balanced score.

**Formula:** F1 = 2 x (Precision x Recall) / (Precision + Recall)

In [ ]:
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)
print(f"F1 Score (manual):  {f1_manual:.4f}")

f1_sklearn = f1_score(y_true_clf, y_pred_clf)
print(f"F1 Score (sklearn): {f1_sklearn:.4f}")

print()
# Demonstrate why harmonic mean is used instead of simple average
p_example, r_example = 0.95, 0.10
simple_avg = (p_example + r_example) / 2
harmonic_mean = 2 * (p_example * r_example) / (p_example + r_example)
print(f"Example with precision={p_example}, recall={r_example}:")
print(f"  Simple average: {simple_avg:.4f}  -- looks moderate, misleading")
print(f"  Harmonic mean (F1): {harmonic_mean:.4f}  -- correctly reflects poor recall")

### 2.6 Specificity (True Negative Rate)

**Definition:** Of all instances that were actually negative, how many did the model correctly identify.

**Formula:** Specificity = TN / (TN + FP)

In [ ]:
specificity_manual = tn / (tn + fp)
print(f"Specificity (manual): {specificity_manual:.4f}")
print()
print("Note: sklearn has no direct specificity_score function.")
print("It can be obtained from the confusion matrix, or as recall of the negative class:")
print(classification_report(y_true_clf, y_pred_clf, target_names=['Negative', 'Positive']))

### 2.7 ROC Curve and AUC

**ROC Curve:** Plots True Positive Rate against False Positive Rate at varying thresholds.

**AUC:** Area under that curve, summarizing separability into a single number.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true_clf, y_prob_clf)
auc_score = roc_auc_score(y_true_clf, y_prob_clf)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC: {auc_score:.4f}")
print()
print("Interpretation:")
print("  AUC = 1.0  -- perfect separation between classes")
print("  AUC = 0.5  -- no better than random guessing")
print("  AUC < 0.5  -- worse than random; predictions may be inverted")

### 2.8 Log Loss (Cross-Entropy Loss)

**Definition:** Evaluates the quality of predicted probabilities. Heavily penalizes confident, incorrect predictions.

**Formula:** LogLoss = -(1/n) x sum(y x log(p) + (1-y) x log(1-p))

In [ ]:
logloss_sklearn = log_loss(y_true_clf, y_prob_clf)
print(f"Log Loss: {logloss_sklearn:.4f}")
print()

# Demonstrate the penalty for confident wrong predictions
print("Demonstration -- same final classification, different confidence:")
case_a_true, case_a_prob = [1], [0.51]   # barely correct, low confidence
case_b_true, case_b_prob = [1], [0.99]   # correct, high confidence
case_c_true, case_c_prob = [1], [0.01]   # confidently WRONG

print(f"  Low-confidence correct (p=0.51):   log loss = {log_loss(case_a_true, case_a_prob, labels=[0,1]):.4f}")
print(f"  High-confidence correct (p=0.99):  log loss = {log_loss(case_b_true, case_b_prob, labels=[0,1]):.4f}")
print(f"  High-confidence WRONG (p=0.01):    log loss = {log_loss(case_c_true, case_c_prob, labels=[0,1]):.4f}")
print()
print("Notice the third case is penalized far more heavily, even though")
print("accuracy alone would not distinguish between a wrong prediction at 0.49 vs 0.01.")

### 2.9 Classification Metrics Summary

In [ ]:
print(classification_report(y_true_clf, y_pred_clf, target_names=['Negative', 'Positive']))

## The Precision-Recall Tradeoff

As the classification threshold changes, precision and recall move in opposite directions. This section visualizes that tradeoff directly.


In [ ]:
thresholds_range = np.arange(0.05, 1.0, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds_range:
    preds_at_t = (y_prob_clf >= t).astype(int)
    precisions.append(precision_score(y_true_clf, preds_at_t, zero_division=0))
    recalls.append(recall_score(y_true_clf, preds_at_t, zero_division=0))
    f1s.append(f1_score(y_true_clf, preds_at_t, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds_range, precisions, 'o-', label='Precision', color='steelblue')
plt.plot(thresholds_range, recalls, 's-', label='Recall', color='coral')
plt.plot(thresholds_range, f1s, '^-', label='F1 Score', color='seagreen')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.6, label='Default threshold (0.5)')
plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('Precision, Recall, and F1 vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Raising the threshold increases precision but decreases recall.")
print("Lowering the threshold increases recall but decreases precision.")
print("The correct threshold depends on which type of error is more costly for the problem.")

## Bias-Variance Decomposition

This section demonstrates the difference between high-bias (underfitting) and high-variance (overfitting) models using a simple polynomial regression example.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# Generate synthetic non-linear data
np.random.seed(42)
X_demo = np.sort(np.random.rand(60, 1) * 10, axis=0)
y_demo = np.sin(X_demo).ravel() + np.random.normal(0, 0.2, X_demo.shape[0])

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.3, random_state=42
)

degrees = [1, 4, 15]  # underfit, good fit, overfit
labels = ['Degree 1 (High Bias)', 'Degree 4 (Balanced)', 'Degree 15 (High Variance)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
X_plot = np.linspace(0, 10, 200).reshape(-1, 1)

for i, (deg, label) in enumerate(zip(degrees, labels)):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train_d, y_train_d)

    train_pred = model.predict(X_train_d)
    test_pred = model.predict(X_test_d)
    train_rmse = np.sqrt(mean_squared_error(y_train_d, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_d, test_pred))

    axes[i].scatter(X_train_d, y_train_d, color='steelblue', s=20, label='Train data')
    axes[i].scatter(X_test_d, y_test_d, color='coral', s=20, label='Test data')
    axes[i].plot(X_plot, model.predict(X_plot), color='black', linewidth=2)
    axes[i].set_title(f'{label}\nTrain RMSE={train_rmse:.3f}  Test RMSE={test_rmse:.3f}',
                       fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_ylim(-2, 2)

plt.suptitle('Bias-Variance Tradeoff: Underfitting vs Good Fit vs Overfitting', fontweight='bold')
plt.tight_layout()
plt.show()

print("Degree 1: high train and test error -- underfitting (high bias)")
print("Degree 4: low train and test error, small gap -- good generalization")
print("Degree 15: very low train error, high test error -- overfitting (high variance)")

## Cross-Validation Error

A more reliable estimate of model performance than a single train-test split.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

model_cv = LinearRegression()

cv_scores = cross_val_score(model_cv, X_demo, y_demo, cv=5, scoring='neg_root_mean_squared_error')
cv_rmse = -cv_scores

print("RMSE across 5 folds:")
for i, score in enumerate(cv_rmse, 1):
    print(f"  Fold {i}: {score:.4f}")

print()
print(f"Mean RMSE: {cv_rmse.mean():.4f}")
print(f"Std Dev:   {cv_rmse.std():.4f}")
print()
print("A low standard deviation across folds indicates the model's performance")
print("is stable and not overly dependent on which specific data points were used for training.")

## Decision Reference

A quick lookup for which metric to use in a given situation.


In [ ]:
decision_guide = pd.DataFrame({
    'Situation': [
        'Regression, no major outliers',
        'Regression, outliers present',
        'Need to explain variance captured',
        'Classification, balanced classes',
        'Classification, imbalanced classes',
        'False positives more costly',
        'False negatives more costly',
        'Comparing models across thresholds',
        'Predicted probabilities must be well-calibrated'
    ],
    'Recommended Metric': [
        'RMSE or MAE',
        'MAE (less sensitive than RMSE)',
        'R-squared',
        'Accuracy',
        'F1 Score, Precision, Recall',
        'Precision',
        'Recall',
        'ROC-AUC',
        'Log Loss'
    ]
})
print(decision_guide.to_string(index=False))

## Summary

Regression metrics: MAE, MSE, RMSE, R-squared, MAPE -- with manual computation and sklearn equivalents.

Classification metrics: confusion matrix, accuracy, precision, recall, F1 score, specificity, ROC-AUC, log loss -- with manual computation and sklearn equivalents.

The precision-recall tradeoff, visualized directly across thresholds.

Bias-variance decomposition, demonstrated with a polynomial regression example at three different complexity levels.

Cross-validation error as a more reliable performance estimate than a single train-test split.

A decision reference table for selecting the correct metric based on the situation rather than convenience.
